# 🗣️ Language Tutor — built on the Claude Agent SDK

### The rise of the agent harness

A raw LLM only predicts text. An **agent harness** is the machinery that turns a model into something that *acts*: a loop that calls tools, manages context and memory, enforces permissions, runs hooks, and delegates to subagents.

The **Claude Agent SDK** is that harness, as a library — the very same engine that powers Claude Code, exposed through one function: `query()`.

To show it off, we'll build a **language tutor**. It starts as ~10 lines and grows, one harness capability at a time, into a real app:

1. **The simplest loop** — a model with a persona
2. **File tools** — it records the vocab, grammar and mistakes it sees
3. **A subagent** — it plans and revises your learning journey
4. **A Gradio app** — vocab, grammar, journey and XP, live

The tutoring intelligence never changes — it's the same model throughout. What changes is everything the *harness* lets us build around it.

> **Auth:** no API key needed. The SDK drives the `claude` CLI, so it uses your **Claude subscription login** — you're already set if you use Claude Code. (On a fresh machine, run `claude setup-token` once.) Just make sure `ANTHROPIC_API_KEY` is *not* set, or it would bill the API instead of your subscription.

## Step 1 · The simplest possible agent loop

This is the whole thing: a persona in the system prompt, one `query()`, and we stream back the reply. No tools, no memory — just a friendly conversation partner.

Notice there's no tool-call loop to write. `query()` *is* the agent loop.

In [1]:
from claude_agent_sdk import query, ClaudeAgentOptions, AssistantMessage, TextBlock

LANGUAGE = "Spanish"

options = ClaudeAgentOptions(
    system_prompt=f"You are speaking with a beginner at {LANGUAGE} and having a friendly conversation to help them learn",
)

async for message in query(prompt="Hola!", options=options):
    if isinstance(message, AssistantMessage):
        for block in message.content:
            if isinstance(block, TextBlock):
                print(block.text)

¡Hola! 👋 ¡Qué bueno verte!

I'm here to help you practice Spanish. ¡Vamos a empezar! (Let's begin!)

How are you today? In Spanish, I might ask:

**"¿Cómo estás?"** (How are you?)

You can try answering with:
- **"Estoy bien"** (I'm good)
- **"Estoy cansado/cansada"** (I'm tired)
- **"Más o menos"** (So-so)

Don't worry about making mistakes — that's how we learn! 😊

¿Cómo estás hoy?


## Step 2 · Give it memory with file tools

Now we hand the tutor the built-in **Read / Write / Edit** tools and point them at a `data/` folder. With a few lines of prompting, it keeps its own notes about you — vocab, grammar, mistakes, who you are, and a learning plan — as plain files.

The options that matter here:
- `model='sonnet'` + `effort='low'` — a fast model at low reasoning effort, for snappy, conversational replies
- `allowed_tools` + `permission_mode='acceptEdits'` — let it write without interrupting you for approval
- `cwd` — scope its files to `data/`
- `setting_sources=[]` — isolate it from your local Claude Code config, so it's *only* the tutor

These files become the tutor's long-term memory — durable across runs, and (next) readable by a UI.

In [ ]:
from pathlib import Path

DATA = Path("data")
DATA.mkdir(exist_ok=True)

TUTOR_PROMPT = f"""You are a warm, patient {LANGUAGE} tutor having a relaxed, friendly conversation with a beginner.
Keep it effortless and fun. Speak mostly in {LANGUAGE}, adding short English glosses for anything new so the learner is never lost. Start very simple and stretch them only a little at a time.

You keep your memory of the learner as files in your working directory. Read any that already exist to recall who you're talking to, and after each exchange update them. Use exactly these shapes:

- vocab.json    -> {{"words": [{{"es": "...", "en": "...", "confidence": 0-5, "count": <times the learner has used it>}}]}}
- grammar.json  -> {{"topics": [{{"topic": "...", "status": "introduced|practising|comfortable", "note": "..."}}]}}
- mistakes.json -> {{"mistakes": [{{"error": "...", "correction": "...", "note": "..."}}]}}
- notes_about_student.md      -> free-form notes: their name, interests, goals
- learning_journey_phases.md  -> your phased plan, and where they are now

When the learner uses a word, add it if new and bump its count; raise its confidence (0 = brand new, 5 = mastered) as they show they know it.
Maintain the files quietly in the background. Never mention them or narrate that you are reading or writing them -- just keep the conversation flowing naturally."""

options = ClaudeAgentOptions(
    system_prompt=TUTOR_PROMPT,
    model="sonnet",        # fast model -> snappy conversation (the SDK default is heavier)
    effort="low",          # low reasoning effort keeps replies quick and chatty
    allowed_tools=["Read", "Write", "Edit"],
    permission_mode="acceptEdits",
    cwd=str(DATA.resolve()),
    setting_sources=[],    # isolate from local Claude Code config -- this agent is only our tutor
)

async for message in query(prompt="¡Hola! Me llamo Ed. Me gustan los perros.", options=options):
    if isinstance(message, AssistantMessage):
        for block in message.content:
            if isinstance(block, TextBlock):
                print(block.text)

In [3]:
# The tutor's memory -- written by the agent itself
for f in sorted(DATA.glob("*")):
    print()
    print(f"===== {f.name} =====")
    print(f.read_text())

## Context & compaction · the harness handles it

**Does history pile up forever?** Not here. Each `query()` call above is its own session — history does *not* carry between cells. The tutor's memory is the **files**, not the chat log, so every turn starts lean.

**What about a long, continuous chat** (the Gradio app later, via `ClaudeSDKClient`)? Then history *does* accumulate within the session — and when it nears the context-window limit, **the SDK compacts it automatically**: it summarizes older turns while keeping the recent ones, and emits a `compact_boundary` system message when it does. Out of the box, nothing to wire up.

You can still steer it when you need to:
- **`PreCompact` hook** — run code before compaction (e.g. archive the full transcript)
- **CLAUDE.md** — add notes on what to preserve when summarizing
- **`/compact`** — send it as a prompt string to compact on demand
- **`max_turns` / `max_budget_usd`** — hard caps on any single run

The nicest part for a tutor: because progress lives in **files on disk**, compaction — or even a brand-new session tomorrow — never loses your learning. The conversation is disposable; the memory is durable.

## Streaming · watch the agent work

Two upgrades to the display:

1. **Token streaming** — set `include_partial_messages=True`, and `query()` also yields `StreamEvent`s carrying the raw API stream. We pull the text deltas (`content_block_delta` → `text_delta`) and print them live.
2. **Interim vs final** — the *final answer* is the streamed text; the *interim work* is the `ToolUseBlock`s (reading and writing your files). Rendering those on their own lines lets you watch the agent loop in action: read your notes → reply → save what's new.

`dataclasses.replace` clones our Step 2 `options` with streaming switched on.

In [ ]:
from pathlib import Path
from claude_agent_sdk import query, AssistantMessage, StreamEvent, ToolUseBlock

async def stream_chat(prompt, options):
    """Stream the reply token-by-token, and show tool activity as it happens."""
    async for message in query(prompt=prompt, options=options):
        if isinstance(message, StreamEvent):
            event = message.event
            if event.get("type") == "content_block_delta":
                delta = event.get("delta", {})
                if delta.get("type") == "text_delta":
                    print(delta["text"], end="", flush=True)
        elif isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, ToolUseBlock):
                    name = Path(block.input.get("file_path", "")).name
                    print()
                    print(f"  · {block.name} {name}".rstrip())
    print()

In [ ]:
from dataclasses import replace

stream_options = replace(options, include_partial_messages=True)
await stream_chat("También tengo dos gatos. Me encanta cocinar. ¿Cómo estás?", stream_options)

## Step 3 · A background journey-coach (a specialist agent)

The tutor stays light and fast (Sonnet, low effort). But planning a good *curriculum* — reading all the progress, spotting patterns in the mistakes, organizing the next phases — is deeper work. So we hand that to a **specialist agent**: the `journey-coach`, running at **high effort**.

The trick: we launch it as an **`asyncio` background task**, so it thinks hard about your learning journey *while the conversation carries on*. The chat never waits for it — watch the timing below: the reply lands in a couple of seconds, the coach keeps working for a minute.

This is orchestration by *your code* — a core reason to reach for the SDK over the CLI: your program decides when to run a deeper agent alongside the conversation. The coach owns `learning_journey_phases.md`; the tutor just keeps light notes.

> The SDK also has *native* subagents (`agents=` + the `Agent` tool, with a `background` flag) for when you want the tutor to delegate on its own. Driving the coach from our code is simpler here and gives us full control over when it runs.

In [ ]:
COACH_PROMPT = f"""You are a {LANGUAGE} curriculum coach working quietly behind the scenes.
Read the learner's files: vocab.json, grammar.json, mistakes.json, notes_about_student.md, and the current learning_journey_phases.md.
Then rewrite learning_journey_phases.md as a short, gently progressive plan in numbered phases -- each with a focus, a few target
words/structures, and a 'ready when' note -- tuned to the learner's recent mistakes and current level. Keep it encouraging and
realistic. Work silently and just update the file."""

coach_options = ClaudeAgentOptions(
    system_prompt=COACH_PROMPT,
    model="sonnet",
    effort="high",                       # think hard about the curriculum
    allowed_tools=["Read", "Write", "Edit"],
    permission_mode="acceptEdits",
    cwd=str(DATA.resolve()),
    setting_sources=[],
)

async def refine_journey():
    """Run the journey-coach to review progress and rewrite the learning journey."""
    async for _ in query(prompt="Review the learner's progress and refine learning_journey_phases.md.", options=coach_options):
        pass

In [ ]:
import asyncio, time

t = time.time()
coach = asyncio.create_task(refine_journey())          # high-effort coach, launched in the background

await stream_chat("Mis gatos se llaman Mango y Salsa.", stream_options)   # the conversation carries on
print(f"[chat done at {time.time()-t:.0f}s -- coach still working: {not coach.done()}]")

await coach                                            # collect the coach whenever we're ready
print(f"[coach done at {time.time()-t:.0f}s]")

In [ ]:
# The journey the coach just rewrote -- at high effort, in the background:
print(Path("data/learning_journey_phases.md").read_text())

## Step 4 · The Gradio app

Now it becomes a *product*. To keep this notebook clean, the whole app lives in three small modules:

- `tutor_core.py` — the engine: the tutor and coach agents, session-resumed streaming, and the `data/` files
- `tutor_app.py` — the Gradio UI: chat on the left; live XP, vocabulary, grammar and learning-journey panels on the right
- `tutor_style.py` — the CSS and JS (palette, the XP bar, the typing dots)

It's the same agent loop from Steps 1–3 — just wrapped in an interface a learner can use. XP is a *view* over what the agent already tracks (`confidence` × 10 + `count`); the journey-coach refines the plan in the background after each turn.

This is the payoff of the SDK over the CLI: we own the whole experience — what streams, what's shown, what runs in the background — for a learner who never sees a terminal.

In [ ]:
# The whole app lives in tutor_core.py (engine), tutor_app.py (UI) and tutor_style.py (CSS/JS).
from tutor_app import launch

launch()